# NGSO SLS - Slice A: Coverage Explorer

Interactive coverage/availability analysis for NGSO constellations (analytic Kepler+J2, H3 grid, single-owner sharding). Defaults to the Reliance-Jio 1600-sat dual shell over India. Runs in **Google Colab** (clones the repo) and **local Jupyter Lab**.

**Outputs:** an interactive geographic coverage map (Plotly), satellites-in-view vs latitude, an availability histogram, and a `coverage_availability.csv` export.

In [ ]:
# === Setup: make ngso_sls importable (Colab + local Jupyter Lab) ===
# LOCAL JUPYTER (recommended): in a terminal, `pip install -e .` in the repo once, then this
#   cell is a no-op (it detects ngso_sls and skips clone/install entirely).
# COLAB: set REPO_URL to your remote. Private repo -> add a GitHub token in Colab 'Secrets'
#   named GITHUB_TOKEN (enable Notebook access).
# NOTE: after you push new code, do Runtime -> Restart runtime, then Run all — a running kernel
#   keeps already-imported modules, so code changes only take effect on a fresh kernel.
REPO_URL = "https://github.com/luca-aalyria/spacetime-sls.git"

import importlib, importlib.util, subprocess, sys, os, re


def _run(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print("$", cmd)
        print(p.stdout[-2000:])
        print(p.stderr[-3000:])
        raise RuntimeError(f"command failed (exit {p.returncode}) - see output above")


if importlib.util.find_spec("ngso_sls") is None:  # no-op on local Jupyter if already installed
    url = REPO_URL
    try:  # optional private-repo auth via Colab secret GITHUB_TOKEN
        from google.colab import userdata
        _tok = userdata.get("GITHUB_TOKEN")
        if _tok and url.startswith("https://github.com/"):
            url = url.replace("https://", f"https://{_tok}@")
    except Exception:
        pass
    repo_dir = re.sub(r"\.git$", "", os.path.basename(REPO_URL.rstrip("/"))) or "repo"
    if not os.path.isdir(repo_dir):
        _run(f"git clone {url} {repo_dir}")
    else:
        # update a pre-existing clone so a stale checkout can't shadow the latest push
        _run(f"git -C {repo_dir} pull --ff-only")
    _run(f"{sys.executable} -m pip install {os.path.abspath(repo_dir)}")
    sys.path.insert(0, os.path.abspath(repo_dir))
    importlib.invalidate_caches()

import ngso_sls
print("ngso_sls", ngso_sls.__version__)

## Controls
Adjust parameters and click **Run simulation**. Sharding gives the same result faster (bit-identical to the unsharded run). `Global` at high H3 resolution is heavy - start coarse.

In [ ]:
# === Interactive control panel ===
# Set parameters and click "Run simulation". (An initial run with defaults renders automatically,
# so "Run all" produces output without clicking.) Requires ipywidgets (bundled in Colab).
from dataclasses import replace
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import ipywidgets as w
from IPython.display import display, clear_output

from ngso_sls.presets import jio_constellation
from ngso_sls.config import Shell, Constellation, TimeGrid, SimConfig
from ngso_sls.pipeline import run_coverage_h3
from ngso_sls.grids.aor import AORS
from ngso_sls.viz.plots import (
    plot_availability_map,
    plot_sats_in_view_vs_latitude,
    plot_availability_hist,
)
from ngso_sls.io.csv_io import write_availability_csv

_style = {"description_width": "110px"}
_L = w.Layout(width="330px")
constellation = w.Dropdown(options=["Jio (1600)", "Custom"], value="Jio (1600)",
                           description="Constellation", style=_style, layout=_L)
aor = w.Dropdown(options=list(AORS), value="India", description="Service area", style=_style, layout=_L)
altitude = w.FloatSlider(value=650, min=300, max=1500, step=10, description="Altitude km", style=_style, layout=_L)
inclination = w.FloatSlider(value=53, min=0, max=90, step=1, description="Inclination deg (Custom)", style=_style, layout=_L)
min_elev = w.FloatSlider(value=25, min=5, max=45, step=1, description="Min elev deg", style=_style, layout=_L)
cell_res = w.IntSlider(value=3, min=1, max=5, description="H3 resolution", style=_style, layout=_L)
duration_min = w.FloatSlider(value=60, min=10, max=240, step=10, description="Duration min", style=_style, layout=_L)
step_s = w.FloatSlider(value=60, min=10, max=120, step=10, description="Time step s", style=_style, layout=_L)
k_cov = w.IntSlider(value=1, min=1, max=4, description="k-coverage", style=_style, layout=_L)
use_shard = w.Checkbox(value=True, description="Use sharding (faster, same result)")
run_btn = w.Button(description="Run simulation", button_style="primary", icon="play")
out = w.Output()


def _simulate(_=None):
    with out:
        clear_output(wait=True)
        if constellation.value.startswith("Jio"):
            shells = tuple(replace(s, min_elev_user_deg=min_elev.value)
                           for s in jio_constellation().shells)
        else:
            shells = (Shell("custom", 600, 20, 1, altitude.value, inclination.value,
                            min_elev_user_deg=min_elev.value),)
        sim = SimConfig(
            Constellation(shells),
            TimeGrid(datetime(2026, 1, 1, tzinfo=timezone.utc),
                     duration_s=duration_min.value * 60.0, step_s=step_s.value),
            k_coverage=k_cov.value,
        )
        print(f"Running {constellation.value} over {aor.value} "
              f"(H3 res {cell_res.value}, {duration_min.value:.0f} min @ {step_s.value:.0f}s, "
              f"k={k_cov.value})...")
        res = run_coverage_h3(
            sim, AORS[aor.value], cell_res=cell_res.value,
            shard_res=(1 if use_shard.value else None), chunk_steps=10,
        )
        a = res["availability"]
        print(f"cells={len(res['cells'])}  availability mean={a.mean():.3f} "
              f"min={a.min():.3f} max={a.max():.3f}  "
              f"mean sats-in-view={res['sats_in_view_mean'].mean():.1f}")
        write_availability_csv(
            res, "coverage_availability.csv",
            {"constellation": constellation.value, "aor": aor.value, "seed": sim.seed,
             "cell_layout": sim.cell_layout, "step_s": sim.time_grid.step_s, "propagator": "KeplerJ2"},
        )
        print("wrote coverage_availability.csv")
        plot_availability_map(res, title=f"Coverage availability - {aor.value}").show()
        plot_sats_in_view_vs_latitude(res)
        plt.show()
        plot_availability_hist(res)
        plt.show()


run_btn.on_click(_simulate)
display(w.VBox([
    w.HBox([constellation, aor]),
    w.HBox([altitude, inclination]),
    w.HBox([min_elev, cell_res]),
    w.HBox([duration_min, step_s]),
    w.HBox([k_cov, use_shard]),
    run_btn, out,
]))
_simulate()  # initial run so "Run all" shows output without clicking